In [ ]:
%cd ../.
import os, sys
sys.path.insert(0, os.path.abspath('../Scripts'))
sys.path.insert(0, os.path.expanduser('~/CDD_Vault_API/python'))  # CDD Vault API (get_df)

In [ ]:
%load_ext autoreload
%autoreload 2

import re
import os
import pandas as pd
import numpy as np
import py3Dmol
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import csv
import pickle

from pathlib import Path
from rdkit import Chem
from rdkit.Chem import AllChem,rdFMCS
# import prolif as plf
from glob import glob
import meeko
import subprocess as sub
# from vina import Vina
import time
from tqdm.auto import tqdm  # auto -> notebook/VSCode widget
tqdm.pandas()
import importlib
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import f1_score, silhouette_score, davies_bouldin_score, calinski_harabasz_score
from xgboost import XGBClassifier,XGBRegressor
from openTSNE import TSNE as oTSNE        # pip install openTSNE
import seaborn as sns
from scipy import stats
import pyarrow.dataset as pads
import csv, contextlib, threading, joblib
import joblib
from joblib import Parallel, delayed
from datetime import date

# user defined modules
import Rdkit_tools as rdkit_tools
importlib.reload(rdkit_tools)
import Molecule as M
import ML_Reg as ML_Reg
import ML_Class as ML_Class
from MolViz3D import MolViz3D
import Statistics_tools as stats_tools
import python.functions as fn
from get_library import get_df   # CDD Vault collection export
from python.MS_build_ML import PARAMS, DATA, OUTPUT   # CLI classes reused here (%autoreload picks up edits)
# from tdc.multi_pred import DTI

In [ ]:
## params — every YAML key becomes an attribute (params.MS_PATH); Dropbox readers + MLTrail vault bound here
CONFIG = 'config/config.yaml'

params = PARAMS(CONFIG)
params.load_params()
params.setup_dropbox()    # params.dbx / params.dbx_glob per DROPBOX_BACKEND
params.load_registry()    # params.registry (shared MLTrail vault)

## transition shim — keeps the bare YAML names (SERAC_C, MS_PATH, ...) working in the cells below
globals().update({k: v for k, v in params.__dict__.items() if k.isupper()})
registry = params.registry
_dbx = _fbx_open = params.dbx
_fbx_glob = params.dbx_glob

print('> chemlib overwrite:', params.CHEMLIB_OVERWRITE, '| dfraw overwrite:', params.DFRAW_OVERWRITE)
print('> features:', params.FEATURES_TYPE)

## 0. Imports

In [ ]:
%%time
## data — library + proteomics + chemistry features (see python/MS_build_ML.py:DATA)
data = DATA()
data.load_chemical_lib_df(params)                                  # serac_df + validated / devalidated targets
## cm2rm now comes from config CM2RM_PARTS — any subset of contaminants (the CONTAMINANTS csv) |
## control_compounds (CONTROLS) | fbx_independent (Px_Ligase_dependent == 0 or Px_validated_WT == 0)
data.get_contaminants_and_controls(params)                         # parts=(...) here would override the config
data.load_proteomics_data(params)                                  # DFRAW_OVERWRITE -> rebuild from the tranches, else read the parquets
data.compute_features(params)                                      # FEATURES_TYPE -> MF_features (+ MF morgan bits)

## OpenTargets disease scores + the big-pharma-relevant subset (see python/MS_build_ML.py:DATA)
data.load_opentargets(params)   # ot_df, cached to OT_CACHE — delete that file to force a recompute from OT_ROOT
data.filter_pharma_targets()    # ot_pharma, via the module-level PHARMA_PRIORITY_AREAS / PHARMA_DROP_AREAS

## transition shim — bare names used by the cells below
serac_df, MS, df_raw, MF, MF_features = data.serac_df, data.MS, data.df_raw, data.MF, data.MF_features
control_compounds, contaminants, cm2rm = data.control_compounds, data.contaminants, data.cm2rm
fbx_independent = data.fbx_independent
validated_targets, devalidated_targets = data.validated_targets, data.devalidated_targets
ot_df, ot_pharma = data.ot_df, data.ot_pharma
# ot_df.to_csv('data/patent/20260609_OpenTarget_list.csv', sep=',', index=False)

## 1. Predictions

#### 1.1. Predicting non-silent compounds (activity > 0)

In [ ]:
## Predicting silent compounds on validation set:
ML_data = MS.copy()
ML_data['label'] = (ML_data['ndown'] > 0) * 1

# add features
ML_data = pd.merge(ML_data[['compound','label']],MF_features,on='compound')#.drop_duplicates().reset_index(drop=True)

# predict
model = RandomForestClassifier(n_estimators=200, class_weight='balanced', n_jobs=16, random_state=0)
# model=HistGradientBoostingClassifier(learning_rate=0.03, max_depth=6, max_leaf_nodes=15,l2_regularization=0.0, random_state=42)
trained_model, df_pred = ML_Class.run_K_Fold_Xval_Classification(
    ML_data, ID='compound', model=model, folds=5, col_to_rm=['compound', 'label'],
    v='tqdm', ctf=0.5, impute_by_mean=False)

ML_Class.metrics_from_pred_df(df_pred)

In [ ]:
## get PPV from probas
ppv_df = ML_Class.get_PPV_vs_proba(df_pred, npts=40, plot=True,
                                   baseline=df_pred['real_y'].mean(), dpi=80)
plt.title('Single/Low PPV vs cutoff'); plt.show()

In [ ]:
## predicting on temporal test sets:
ML_data = MS.copy()
ML_data['label'] = (ML_data['ndown'] > 0) * 1

## define temporal test set:
date_split = '2026-07-14'
test_cmps  = list(MS[MS['date'] >= date_split]['compound'])
print('> number test compounds:',len(test_cmps))
train_cmps = list(ML_data[~ML_data['compound'].isin(test_cmps)]['compound'])

# add features
ML_data = pd.merge(ML_data[['compound','label']],MF_features,on='compound')

# predict the test compounds via a single defined train/test split (unseen-compound eval)
model=HistGradientBoostingClassifier(learning_rate=0.03, max_depth=6, max_leaf_nodes=15,l2_regularization=0.0, random_state=42)
trained_model, df_pred = ML_Class.K_fold_by_defined_IDs_Classification(
    ML_data, ID='compound', ID_sets=[[train_cmps, test_cmps]], model=model,
    col_to_rm=['compound', 'label'], v=False, ctf=0.5, impute_by_mean=False)

auc_m = ML_Class.metrics_from_pred_df(df_pred)['roc_auc']
print(auc_m)

In [ ]:
## get PPV from probas
ppv_df = ML_Class.get_PPV_vs_proba(df_pred, npts=40, plot=True,baseline=df_pred['real_y'].mean(), dpi=80)
plt.title('Single/Low PPV vs cutoff'); plt.show()

In [ ]:
ppv_df[ppv_df['threshold']>= 0.8]

#### 1.2. Predicting Single-Low by threshold

In [ ]:
## find the best label threshold `end` (ndown in [start, end]).
## the label only changes at actual ndown values, so we fit ONCE per distinct cut-point
## (not per integer end) and expand the curve — identical result, ~10x fewer fits.
## HGBC multithreads each fit across all cores, so we run the (few) cut-points sequentially.
SingleLow_ouput = 'output/SingleLow_analysis/range_1_260_singleTask_RF.csv'

if not os.path.exists(SingleLow_ouput):

    start, max_it = 1, 260
    base = pd.merge(MS[['compound', 'ndown']], MF_features, on='compound')
    feat = base.drop(columns='ndown')                 # compound + features (constant across trials)
    cut_ends = [int(e) for e in np.unique(base['ndown']) if start <= e < max_it]
    print(f'> {len(cut_ends)} distinct ndown cut-points to fit (vs {max_it - start} integer ends)')

    fit_at = {}
    for end in tqdm(cut_ends):
        ML_data = feat.copy()
        ML_data['label'] = ((base['ndown'] >= start) & (base['ndown'] <= end)).astype(int).to_numpy()
        # model = HistGradientBoostingClassifier(learning_rate=0.03, max_depth=6,max_leaf_nodes=15, l2_regularization=0.0, random_state=42)
        model = RandomForestClassifier(n_estimators=400, class_weight='balanced', n_jobs=32, random_state=0)
        _, df_pred = ML_Class.run_K_Fold_Xval_Classification(
            ML_data, ID='compound', model=model, folds=5, col_to_rm=['compound', 'label'],
            v=False, ctf=0.5, impute_by_mean=False)
        m = ML_Class.metrics_from_pred_df(df_pred)
        fit_at[end] = (m['roc_auc'], m['pr_auc'], int(ML_data['label'].sum()))

    ## expand: every integer end reuses the fit at the largest cut-point <= it (same label -> same AUC)
    results, last = [], (np.nan, np.nan, 0)
    for end in range(start, max_it):
        if end in fit_at: last = fit_at[end]
        results.append([f'{start}_{end}', last[0], last[1], last[2]])

    res_df = pd.DataFrame(results, columns=['range', 'auc', 'pr_auc', 'n'])
    res_df.to_csv(SingleLow_ouput, index=False)

else:
    res_df = pd.read_csv(SingleLow_ouput)

In [ ]:
res_df['xl'] = res_df['range'].str.split('_').str[1].astype(int)

# AUC vs upper activity bound (xl), styled like get_PPV_vs_proba
plot_df = res_df[:50]
fig, ax = plt.subplots(figsize=(10, 6), dpi=150)
plt.plot('xl', 'auc',    data=plot_df, color=SERAC_C['azure'], linewidth=2, marker='o', markersize=4, label='ROC AUC')
plt.plot('xl', 'pr_auc', data=plot_df, color=SERAC_C['ember'], linewidth=2, marker='o', markersize=4, label='PR AUC')
plt.axhline(0.5, color='r', linestyle='--', linewidth=1.5, label='chance (0.5)')

# x-ticks as ranges every 5: 1-1, 1-5, 1-10, 1-15, ...
ticks = [1] + list(range(5, int(plot_df['xl'].max()) + 1, 5))
ax.set_xticks(ticks); ax.set_xticklabels([f'1-{t}' for t in ticks], rotation=45)
plt.legend(); plt.xlabel('activity range (ndown in [1, xl])'); plt.ylabel('AUC')
plt.ylim(0.49, 0.9); plt.grid(axis='both', alpha=0.3)

res_df.head(20)

In [ ]:
res_df.loc[res_df.range.isin(['1_8','1_12']), 'n']

In [ ]:
## build activity classifier:
ML_data = MS.copy()
ML_data['label'] = ML_data['ndown'].apply(lambda x: (x >= 1 and x <= 12 ) * 1)
# ML_data = ML_data[ML_data['date'] <= '2026-07-14'] # 
print(ML_data.shape)

# add features
ML_data = pd.merge(ML_data[['compound','label']],MF_features,on='compound')#.drop_duplicates().reset_index(drop=True)

# predict
model = RandomForestClassifier(n_estimators=200, class_weight='balanced', n_jobs=16, random_state=0)
# model=HistGradientBoostingClassifier(learning_rate=0.03, max_depth=6, max_leaf_nodes=15,l2_regularization=0.0, random_state=42)
trained_model, df_pred = ML_Class.run_K_Fold_Xval_Classification(
    ML_data, ID='compound', model=model, folds=5, col_to_rm=['compound', 'label'],
    v='tqdm', ctf=0.5, impute_by_mean=False)
print('classification roc_auc:', round(ML_Class.metrics_from_pred_df(df_pred)['roc_auc'], 3))

# ## ICP (inductive conformal prediction): same CV, adds per-compound class p-values (p_1/p_2) + credibility (cred)
# icp_model, icp_pred = ML_Class.run_K_Fold_Xval_CP(
#     ML_data, ID='compound', model=model, folds=5, col_to_rm=['compound', 'label'],
#     v=False, ctf=0.5, impute=False)
# print('ICP roc_auc:', round(ML_Class.metrics_from_pred_df(icp_pred)['roc_auc'], 3))
# icp_pred.head()

In [ ]:
tmp = ML_Class.plot_roc_curve([df_pred],c=[SERAC_C['azure']],dpi=50)

In [ ]:
# cred_ctf = 0.4
# low_cred  = icp_pred[icp_pred['cred'] <  cred_ctf]
# high_cred = icp_pred[icp_pred['cred'] >= cred_ctf]
# # overlay the original (non-ICP) classifier ROC alongside the two credibility-split ICP curves
# tmp = ML_Class.plot_roc_curve(
#     [df_pred, low_cred, high_cred],
#     l=['all (no ICP)', 'cred < %.2f' % cred_ctf, 'cred >= %.2f' % cred_ctf],
# )

In [ ]:
## get PPV from probas
ppv_df = ML_Class.get_PPV_vs_proba(df_pred, npts=40, plot=True,baseline=df_pred['real_y'].mean(), dpi=150,min_n=20,color=SERAC_C['azure'])
plt.title('Single/Low PPV vs cutoff'); plt.show()
ppv_df[ppv_df['threshold']>=0.75]

In [ ]:
print(MS['date'].unique())

In [ ]:
## predicting on temporal test sets:
ML_data = MS.copy()
ML_data['label'] = ML_data['ndown'].apply(lambda x: (x >= 1 and x <= 12 ) * 1)

## define temporal test set:
date_split = '2026-07-14' # '2026-07-06' # '2026-05-20' #  # '2026-07-14' #       
test_cmps  = list(MS[MS['date'] > date_split]['compound'])
print('> number test compounds:',len(test_cmps))
train_cmps = list(ML_data[~ML_data['compound'].isin(test_cmps)]['compound'])

## make sure no data leakage between train and test (by compound id AND by structure/SMILES)
assert stats_tools.intersection(test_cmps,train_cmps) == []
# also check no test SMILES appears in the training set (same structure under a different id)
_smi = serac_df.set_index('compound')['smiles']
assert _smi[test_cmps].isin(_smi[train_cmps]).sum() == 0

# add features
ML_data = pd.merge(ML_data[['compound','label']],MF_features,on='compound')

# predict the test compounds via a single defined train/test split (unseen-compound eval)
model = RandomForestClassifier(n_estimators=400, class_weight='balanced', n_jobs=16, random_state=0)
# model=HistGradientBoostingClassifier(learning_rate=0.03, max_depth=6, max_leaf_nodes=15,l2_regularization=0.0, random_state=42)
trained_model, df_pred_test = ML_Class.K_fold_by_defined_IDs_Classification(
    ML_data, ID='compound', ID_sets=[[train_cmps, test_cmps]], model=model,
    col_to_rm=['compound', 'label'], v=False, ctf=0.5, impute_by_mean=False)

ML_Class.metrics_from_pred_df(df_pred_test)

In [ ]:
## get PPV from probas
ppv_df = ML_Class.get_PPV_vs_proba(df_pred_test, npts=40, plot=True,baseline=df_pred_test['real_y'].mean(), dpi=150,min_n=11,color=SERAC_C['sky_blue'])
plt.title('Single/Low PPV vs cutoff'); plt.show()

In [ ]:
## check chemical distance vs confidence:
## applicability-domain check (same analysis as MS_ML_Prioritization) — nearest-neighbour Tanimoto
## distance (Rdkit_tools) of each temporal-test compound to the TRAINING set, vs the model probability.
## Uses the cell above: train_cmps / test_cmps / df_pred_test (probas, real_y), and MF (binary Morgan bits).
train_fp = MF[MF['compound'].isin(train_cmps)].drop_duplicates('compound')
test_fp  = MF[MF['compound'].isin(test_cmps)].drop_duplicates('compound')

# per-test nearest-neighbour Tanimoto distance to training (chunked -> memory-safe)
test_nn   = rdkit_tools.compute_NNs_in_chunks(test_fp, train_fp, chunk_size=1000, top=1)
dist_conf = (test_nn.rename(columns={'distance': 'nn_dist'})[['compound', 'nn_dist']]
             .merge(df_pred_test[['compound', 'probas', 'real_y']], on='compound'))

# training's own NN spread (exclude self): Rdkit_tools distance matrix, mask the diagonal
dmat = rdkit_tools.get_RDKiTTanimoto_distance_matrix(train_fp, train_fp).to_numpy().copy()  # writable
np.fill_diagonal(dmat, np.inf)
train_self = dmat.min(axis=1)
edge = np.quantile(train_self, 0.95)                                  # 'edge of domain' reference
frac_out = (dist_conf['nn_dist'] > edge).mean()
rho = stats.spearmanr(dist_conf['nn_dist'], dist_conf['probas']).correlation

print(f"> {len(dist_conf)} test compounds | NN Tanimoto dist "
      f"[{dist_conf.nn_dist.min():.2f}, {dist_conf.nn_dist.max():.2f}] | "
      f"domain edge (train 0.95q)={edge:.2f} | Spearman(dist, proba)={rho:.2f}")
print(f"> {frac_out:.1%} of test sits BEYOND the training's 95th-pct NN distance (out-of-domain)")

## plot: (L) distance distributions, (R) probability vs distance, colored by true label
fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 5), dpi=120)
a1.hist(train_self, bins=40, density=True, alpha=0.6, color=SERAC_C['azure'], label='training <-> training')
a1.hist(dist_conf['nn_dist'], bins=40, density=True, alpha=0.6, color=SERAC_C['ember'], label='test -> training')
a1.axvline(edge, color='k', ls='--', lw=1, label=f'domain edge (0.95q = {edge:.2f})')
a1.set_xlabel('nearest-neighbour Tanimoto distance to training'); a1.set_ylabel('density')
a1.set_title('Chemical-space distance: test vs training'); a1.legend()

sc = a2.scatter(dist_conf['nn_dist'], dist_conf['probas'], c=dist_conf['real_y'],
                cmap='coolwarm', vmin=0, vmax=1, s=25, alpha=0.7, edgecolors='k', linewidths=0.3)
a2.axvline(edge, color='k', ls='--', lw=1, label=f'domain edge ({edge:.2f})')
a2.axhline(0.5, color='grey', ls='--', lw=1, label='decision boundary (0.5)')
fig.colorbar(sc, ax=a2, label='true label (1 = ndown in [1,12])')
a2.set_xlabel('nearest-neighbour Tanimoto distance to training'); a2.set_ylabel('predicted p(ndown in [1,12])')
a2.set_title('Test: model probability vs distance to training'); a2.legend()
plt.tight_layout()

In [ ]:
## visually check top predictions: and show nearest neighbor:
## top-N temporal-test compounds by predicted probability, each paired with its nearest training
## neighbour (from test_nn). Rows alternate TEST (p, true y) | its NN (Tanimoto distance).
TOPN = 20
top = (dist_conf.merge(test_nn[['compound', 'NN', 'distance']], on='compound')
       .sort_values('probas', ascending=False).head(TOPN))
smi = serac_df.drop_duplicates('compound').set_index('compound')['smiles']   # test + train SMILES

# interleave [test, its NN] so mols_per_row=2 puts each pair on one row
rows = []
for _, r in top.iterrows():
    rows.append({'label': f"TEST {r['compound']}  p={r['probas']:.2f}  y={int(r['real_y'])}", 'smiles': smi.get(r['compound'])})
    rows.append({'label': f"NN {r['NN']}  Td={r['distance']:.2f}",                             'smiles': smi.get(r['NN'])})
pairs = pd.DataFrame(rows)

rdkit_tools.draw_compounds_grid(pairs, smiles_col='smiles', compound_col='label',
                                mols_per_row=2, sub_img_size=(320, 260))

In [ ]:
ppv_df[ppv_df['threshold']>=0.75]

In [ ]:
## save/update model to MLTrail:
## deployable binary classifier — "active" = ndown in [1, 12] vs rest, whole library, H236 features.
## MLTrail featurizes new SMILES with its built-in H236 featurizer (vendored bit-for-bit from
## compute_H236_features) and aligns to the saved feature_cols.

# label = activity in [1, 12]; features = the H236 universe already in MF_features
ML_data = MS.copy()
ML_data['label'] = ((ML_data['ndown'] >= 1) & (ML_data['ndown'] <= 12)).astype(int)
ML_data = pd.merge(ML_data[['compound', 'label']], MF_features, on='compound')
feat_cols = [c for c in ML_data.columns if c not in ('compound', 'label')]
print(f'> ML_data: {len(ML_data):,} rows | features={len(feat_cols):,}')
print(ML_data['label'].value_counts().to_string())

clf = HistGradientBoostingClassifier(learning_rate=0.03, max_depth=6, max_leaf_nodes=15,
                                     l2_regularization=0.0, random_state=42)
# clf = RandomForestClassifier(n_estimators=400, class_weight='balanced', n_jobs=16, random_state=0)


# CV metrics for the registry (same 5-fold as the classifier cell above)
_, cv_pred = ML_Class.run_K_Fold_Xval_Classification(
    ML_data, ID='compound', model=clf, folds=5, col_to_rm=['compound', 'label'],
    v=False, ctf=0.5, impute_by_mean=False)
cv = ML_Class.metrics_from_pred_df(cv_pred)
print(f"> CV roc_auc={cv['roc_auc']:.3f}  pr_auc={cv['pr_auc']:.3f}")

# fit on the whole library and hand the {model, feature_cols} bundle straight to MLTrail
# (it joblib-dumps the bundle into the vault as <id>_v<ver>.joblib — no separate local copy needed)
clf.fit(ML_data[feat_cols], ML_data['label'])
bundle = {'model': clf, 'feature_cols': feat_cols, 'task': 'activity_1_12_binary',
          'positive': 'ndown in [1,12]', 'features': 'H236', 'n_train': len(ML_data),
          'class_counts': ML_data['label'].value_counts().to_dict(),
          'sklearn_ver': __import__('sklearn').__version__}

# training set to archive alongside the model — MLTrail keeps only compound/smiles/label
# (features are reconstructible from smiles via the H236 featurizer), stored as delta parquet
train_set = ML_data[['compound', 'label']].merge(
    serac_df[['compound', 'smiles']].drop_duplicates('compound'), on='compound')

# if a model with this name already exists, add a new version to its id (keeps history);
# otherwise register a fresh entry. avoids spawning duplicate ids on re-run.
name = 'Px_activity_1_12'                       # HGBC model — NOT the _rf entry
hit  = registry.list().pipe(lambda d: d.loc[d['experiment_name'] == name, 'id'])
model_id = int(hit.iloc[0]) if len(hit) else None

overwrite = False    # True = reset the latest version in place (use to CORRECT a bad version);
                    # False = append a new version (normal "retrain as data grew")
model_id = registry.add(model_id=model_id, overwrite=overwrite and model_id is not None, model=bundle,
                        experiment_name=name, experiment_measure='proteomics_activity',
                        unit='n_proteins_down', model_type='single_task_classification',
                        framework='sklearn', features_type='H236',
                        training_set=train_set, smiles_column='smiles',
                        compound_id_column='compound', label_column='label',
                        metrics={'roc_auc': round(cv['roc_auc'], 3), 'pr_auc': round(cv['pr_auc'], 3)})
action = 'overwrote latest version of' if (overwrite and len(hit)) else ('new version of' if len(hit) else 'registered')
print(f'> {action} MLTrail model id={model_id}  ({len(train_set):,} training rows archived)')
registry.list()


In [ ]:
## check model details:
for k, v in registry.details(model_id).items():
    print(f'{k:20s}: {v}')

In [ ]:
registry.list()

#### 1.3. Predicting High activity:

In [ ]:
## Predicting silent compounds on validation set:
ML_data = MS.copy() # [MS['date']!='2026-08-13']
ML_data['label'] = (ML_data['ndown'] > 25) * 1
print(f'> {ML_data['label'].sum()} positive samples, ({ML_data['label'].sum()/ML_data.shape[0]}) ')

# add features
ML_data = pd.merge(ML_data[['compound','label']],MF_features,on='compound')#.drop_duplicates().reset_index(drop=True)

# predict
model = RandomForestClassifier(n_estimators=200, class_weight='balanced', n_jobs=16, random_state=0)
# model=HistGradientBoostingClassifier(learning_rate=0.03, max_depth=6, max_leaf_nodes=15,l2_regularization=0.0, random_state=42)
trained_model, df_pred = ML_Class.run_K_Fold_Xval_Classification(
    ML_data, ID='compound', model=model, folds=5, col_to_rm=['compound', 'label'],
    v='tqdm', ctf=0.5, impute_by_mean=False)

tmp = ML_Class.plot_roc_curve([df_pred],c=[SERAC_C['azure']],dpi=150)

ML_Class.metrics_from_pred_df(df_pred)

In [ ]:
## get PPV from probas
ppv_df = ML_Class.get_PPV_vs_proba(df_pred, npts=40, plot=True,baseline=df_pred['real_y'].mean(), dpi=80)
plt.title('Single/Low PPV vs cutoff'); plt.show()

In [20]:
## round trip prediction from MLTrail
# re-read the config and reopen the vault
registry = params.load_params().load_registry().registry
high_id  = int(registry.list().set_index('experiment_name').loc[params.HIGH_MODEL_NAME, 'id'])
bundle   = joblib.load(registry.details(high_id)['model_path']) #_> dict={model,feature_cols,positive,...}

# MS compounds with features and smiles; true label from the bundle's rule: ndown > 25
rt = (MS[['compound', 'ndown']]
        .merge(MF_features[['compound']], on='compound')
        .merge(serac_df[['compound', 'smiles']].drop_duplicates('compound'), on='compound'))
rt['real_y'] = rt.eval(bundle['positive']).astype(int)

# mltrail: featurize the smiles, then predict
pred = registry.predict(high_id, rt[['compound', 'smiles']], smiles_column='smiles',
                        compound_id='compound') #_> df=cols:[smiles,compound,prediction,probability]
rt = rt.merge(pred[['compound', 'prediction', 'probability']]
              .rename(columns={'prediction': 'pred_y', 'probability': 'probas'}), on='compound')

# direct: the same model on MF_features
X = rt[['compound']].merge(MF_features, on='compound')[bundle['feature_cols']]
rt['probas_mf'] = bundle['model'].predict_proba(X)[:, 1]

ML_Class.metrics_from_pred_df(rt) | {'max_proba_diff': (rt['probas'] - rt['probas_mf']).abs().max()}

> loaded 85 params from config/config.yaml


{'roc_auc': 1.0, 'pr_auc': 1.0, 'max_proba_diff': 0.0}